# MTAH 2027 — video-ball fill on a Colab GPU (Jalur 1)

Runs `python -m mtah.vball fill` (top-1 ball detection on every graded frame where the ATD has no
ball, mapped to the pitch with the provided homography) on a GPU instead of the laptop.

Same code, same pinned `ultralytics` / `opencv` as locally. Positions are rounded to 1 cm and
confidences to 1e-3, so the output should be identical to the M1 run. **That is checked, not
assumed:** first run with `LIMIT = 2000` and `TAG = "ft1"`, then compare locally against the
full M1 file (`cache/BAR-ATH_vball_ft1.npz`). Only if identical is a Colab fill used (WORKLOG
2026-11-15, ft2 rule).

Needs on Drive:
- `DRIVE_ROOT/vball/mtah_colab_bundle.zip` (repo code + `cache/BAR-ATH_tactical_atd.npz`,
  built from `outputs/vball/mtah_colab_bundle.zip`)
- the tactical folder (`BAR-ATH_tactical.mp4`, `BAR-ATH_tactical_homography.csv`) at `TACTICAL`
- the weights at `vball/runs/<RUN>/weights/best.pt` (written there by `colab_vball.ipynb`)

Output: `DRIVE_ROOT/vball/fill/BAR-ATH_vball_<OUT_TAG>.npz` -> download to `cache/` locally.

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/Metrica-Hackathon/mtah"
TACTICAL = "/content/drive/MyDrive/Metrica-Hackathon/dataset/tactical"
RUN = "ft1"                 # which weights
TAU = 0.05                  # Stage A choice for that model (ft1: 0.05)
LIMIT = 2000                # 2000 = reproducibility check vs the M1 file; 0 = full match
OUT_TAG = f"{RUN}_colab{LIMIT}" if LIMIT else RUN
WEIGHTS = f"{DRIVE_ROOT}/vball/runs/{RUN}/weights/best.pt"
print(OUT_TAG, WEIGHTS)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
!pip -q install ultralytics==8.4.212 opencv-python-headless==5.0.0.93
import cv2, ultralytics; print(cv2.__version__, ultralytics.__version__)

In [ ]:
# code + ATD cache, and the video on local disk (sequential decode from Drive is slow)
!rm -rf /content/repo && mkdir -p /content/repo /content/data/tactical
!unzip -q "{DRIVE_ROOT}/vball/mtah_colab_bundle.zip" -d /content/repo
!cp "{TACTICAL}/BAR-ATH_tactical.mp4" "{TACTICAL}/BAR-ATH_tactical_homography.csv" /content/data/tactical/
!ls -la /content/data/tactical
import os
os.environ["MTAH_DATA_DIR"] = "/content/data"

In [ ]:
limit = f"--limit {LIMIT}" if LIMIT else ""
!cd /content/repo && python -m mtah.vball fill --weights "{WEIGHTS}" --tag {OUT_TAG} --tau {TAU} \
    --device 0 --batch 32 --save_every 2000 {limit} 2>&1 | grep -v -iE "ultralytics|settings"
!mkdir -p "{DRIVE_ROOT}/vball/fill" && cp /content/repo/cache/BAR-ATH_vball_{OUT_TAG}.npz "{DRIVE_ROOT}/vball/fill/"
!ls -la "{DRIVE_ROOT}/vball/fill/"

If the runtime disconnects during a full fill: re-run the cells; copy
`/content/repo/cache/BAR-ATH_vball_<OUT_TAG>.npz.partial.npz` to Drive beforehand if you want to
resume (the fill resumes from that file).